# PANDA — Final submission notebook

**Settings:** GPU T4 ON, Internet OFF

**Attach on Kaggle:**
1. `prostate-cancer-grade-assessment` (official PANDA competition dataset)
2. `panda-effnetb0-ordinal-5fold` (your private weights)
3. `panda-effnetb0-mse-5fold` (your private weights)
4. `panda-src` (your private src code dataset)
5. `efficientnetpytorch063` by optimo (offline package: https://www.kaggle.com/datasets/optimo/efficientnetpytorch063)

In [ ]:
BATCH_SIZE = 16
ORDINAL_MODE = 'threshold'
N_FOLDS = 5
OUTPUT_CSV = '/kaggle/working/submission.csv'

MODEL_FAMILIES = [
    {
        'name': 'b0_ordinal',
        'weights_dir': '/kaggle/input/datasets/gojamodicsila/panda-effnetb0-ordinal-5fold',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_ordinal_fold{fold}.pth',
        'model_kind': 'baseline',
    },
    {
        'name': 'b0_mse',
        'weights_dir': '/kaggle/input/datasets/gojamodicsila/panda-effnetb0-mse-5fold',
        'backbone': 'efficientnet-b0',
        'weight_pattern': 'efficientnetb0_mse_fold{fold}.pth',
        'model_kind': 'baseline',
    },
]

In [ ]:
import sys
import os
sys.path.insert(0, '/kaggle/input/datasets/optimo/efficientnetpytorch063/efficientnet_pytorch-0.6.3')

In [ ]:
import glob
import shutil
import numpy as np
import pandas as pd
import torch
import cv2
import skimage.io

if not os.path.exists('/kaggle/working/src'):
    shutil.copytree('/kaggle/input/datasets/gojamodicsila/panda-src',
                    '/kaggle/working/src')
    print('Copied src')
else:
    print('src already exists')

sys.path.insert(0, '/kaggle/working')

from src.eval import round_preds
from src.inference import load_model, predict

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# Dataset that reads tiff files directly -- works on real test set
class TiffTestDataset(torch.utils.data.Dataset):
    def __init__(self, df, image_dir):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        path = os.path.join(self.image_dir, f'{row.image_id}.tiff')
        img = skimage.io.MultiImage(path)[-1]
        img = cv2.resize(img, (512, 512))
        img = img.astype(np.float32) / 255.0
        mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
        std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
        img = (img - mean) / std
        img = np.ascontiguousarray(img.transpose(2, 0, 1))
        return torch.from_numpy(img), torch.tensor(0.0)

In [ ]:
TEST_DIR = None
for candidate in [
    '/kaggle/input/prostate-cancer-grade-assessment/test_images',
    '/kaggle/input/competitions/prostate-cancer-grade-assessment/test_images',
]:
    if os.path.exists(candidate):
        TEST_DIR = candidate
        break

TEST_CSV = None
for candidate in [
    '/kaggle/input/prostate-cancer-grade-assessment/test.csv',
    '/kaggle/input/competitions/prostate-cancer-grade-assessment/test.csv',
]:
    if os.path.exists(candidate):
        TEST_CSV = candidate
        break

print('TEST_DIR:', TEST_DIR)
print('TEST_CSV:', TEST_CSV)

test_df = pd.read_csv(TEST_CSV)
test_df['isup_grade'] = 0
print('Test slides:', len(test_df))

if TEST_DIR is not None and os.path.exists(TEST_DIR):
    test_dataset = TiffTestDataset(test_df, TEST_DIR)
    all_family_preds = []
    for family in MODEL_FAMILIES:
        fold_preds = []
        for fold in range(N_FOLDS):
            weight_path = os.path.join(
                family['weights_dir'],
                family['weight_pattern'].format(fold=fold)
            )
            model = load_model(
                weight_path,
                backbone=family.get('backbone', 'efficientnet-b0'),
                device=device,
                model_kind=family.get('model_kind', 'baseline'),
            )
            preds = predict(
                model, test_dataset, device,
                batch_size=BATCH_SIZE,
                ordinal_mode=ORDINAL_MODE,
            )
            fold_preds.append(preds)
            del model
            if device.type == 'cuda':
                torch.cuda.empty_cache()
            print(f"{family['name']} fold {fold} done")
        all_family_preds.append(np.mean(fold_preds, axis=0))
    ensemble_preds = np.mean(all_family_preds, axis=0)
    final_preds = round_preds(ensemble_preds)
else:
    print('Test images not found - saving dummy submission')
    final_preds = np.zeros(len(test_df), dtype=int)

submission = pd.DataFrame({
    'image_id': test_df.image_id.values,
    'isup_grade': final_preds,
})
submission.to_csv(OUTPUT_CSV, index=False)
print('Saved:', OUTPUT_CSV)
print(submission.head())